# BoneNano step 2 — preprocess the projections

The 95 GB pass. Streams scan 86 through the same chain this folder's
`step0a_convert` uses, and writes `/exchange/pdata`.

```
dark_subtract_clip -> fill_dead -> remove_outliers -> flat_field_divide -> bin
```

**This is slow.** At the default `ntheta=500, stride=8` it reads 500 full
2592 x 4608 frames; the full 4001 is eight times that. Written straight into
the HDF5 chunk by chunk rather than held in RAM.

`pdata` is **INTENSITY**. `RecCoded.F0` compares `K|psi|^2` against it and the
amplitude model takes its own square root, so nothing is square-rooted here.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../../src'))
import numpy as np, cupy as cp, h5py
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

def cbshow(ax, img, title, cmap='gray', **kw):
    im = ax.imshow(np.asarray(img), cmap=cmap, **kw)
    ax.set_title(title, fontsize=9)
    cax = make_axes_locatable(ax).append_axes('right', size='5%', pad=0.05)
    ax.figure.colorbar(im, cax=cax)

print('GPU:', cp.cuda.runtime.getDeviceProperties(0)['name'].decode())

## Config

In [ ]:
# --- everything comes from config_step6.conf. Edit THAT file, not this cell.
from holotomocupy.config import parse_args_coded
CFG = parse_args_coded('config_step6.conf')

RAW_DIR = '/data3/cekmekci/bone_nano_holotomo'   # raw_dir; parse_args_coded has no key for it
OUT_H5  = CFG.in_file

DARK_H5  = f'{RAW_DIR}/scan-0067_orca.h5'   # dark
FLAT_H5  = f'{RAW_DIR}/scan-0068_orca.h5'   # flat: no CA1, no CA2
GRID_H5  = f'{RAW_DIR}/scan-0086_orca.h5'   # CA1 + CA2 + rotating sample, 95 GB
META_H5  = f'{RAW_DIR}/scan-0086.h5'        # companion: rotation angle
ORCA     = '/entry/instrument/orca/data'
THETA_DS = '/entry/instrument/tom_ry/value'   # DEGREES

ENERGY  = CFG.energy
Z1      = CFG.z1                 # focus -> SAMPLE
Z_CODE  = CFG.z1_code            # focus -> CA2
Z_DET   = CFG.focusdetectordistance  # focus -> detector  (TODO: confirm per beamtime)
DET_PIX = CFG.detector_pixelsize # UNBINNED
nz, n   = CFG.crop_nz, CFG.crop_n   # NATIVE crop resolution -- NOT CFG.nz/CFG.n,
                                    # which are step 6's own BINNED solve grid.
                                    # Steps 1-5 never bin anything; that is
                                    # CodedReader's job alone at step 6 (same
                                    # as AtomiumS2_coded). See the README's
                                    # "steps 1-5 never bin" section.
CROP    = (nz, n)
DEZINGER = (3, 0.8)
PREVIEW_BIN = CFG.bin   # cosmetic ONLY -- shrinks a notebook thumbnail in
                        # PDATA_CHECK below. Not used in any registration or
                        # writing math; see the note below.

# NOT read for any registration/writing math: CFG.ntheta/start_theta/stride
# (preprocessing never subsamples the angles -- see GEOM_THETA below) or
# CFG.bin/CFG.nz/CFG.n (preprocessing never bins -- see above). All four are
# the step6.py/CodedReader read-time parameters, not preprocessing's.

print(f'{ENERGY} keV   z1(sample) {Z1} m   z1_code(CA2) {Z_CODE} m   z_det {Z_DET} m')
print(f'crop {CROP}, native resolution -- step 6 bins to {CFG.nz} x {CFG.n} at bin={CFG.bin}')
assert Z_CODE < Z1, 'CA2 must be BEFORE the sample for RecCoded'

## Preprocessing helpers

In [ ]:
# Ports of phascade.preprocess.{dark_subtract_clip, fill_dead, remove_outliers,
# flat_field_divide}. Identical to this folder's own step0a_convert -- same
# detector, same chain. The two stages MUST preprocess the same way, or the
# warm start from stage 0 does not line up with the data here.
from cupyx.scipy.ndimage import median_filter

def dark_subtract_clip(data, dark_mean):
    out = np.asarray(data, 'float32') - dark_mean[np.newaxis]
    np.clip(out, 0, None, out=out)
    return out

def _runs(idx):
    return [] if len(idx) == 0 else np.split(idx, np.where(np.diff(idx) > 1)[0] + 1)

def fill_dead(data, mask):
    out = np.array(data, copy=True)
    ny, nx = mask.shape
    for g in _runs(np.where(mask.mean(1) > 0.5)[0]):
        lo, hi = g[0] - 1, g[-1] + 1
        hl, hh = lo >= 0, hi < ny
        for r in g:
            if hl and hh:
                w = (r - lo) / (hi - lo)
                out[..., r, :] = (1 - w) * out[..., lo, :] + w * out[..., hi, :]
            elif hl:
                out[..., r, :] = out[..., lo, :]
            elif hh:
                out[..., r, :] = out[..., hi, :]
    for g in _runs(np.where(mask.mean(0) > 0.5)[0]):
        lo, hi = g[0] - 1, g[-1] + 1
        hl, hh = lo >= 0, hi < nx
        for c in g:
            if hl and hh:
                w = (c - lo) / (hi - lo)
                out[..., :, c] = (1 - w) * out[..., :, lo] + w * out[..., :, hi]
            elif hl:
                out[..., :, c] = out[..., :, lo]
            elif hh:
                out[..., :, c] = out[..., :, hi]
    return out

def remove_outliers(data, radius=3, threshold=0.8):
    # window is radius x radius, NOT 2*radius+1 -- matches phascade
    res = np.asarray(data, 'float32').copy()
    for k in range(res.shape[0]):
        f = cp.asarray(res[k])
        med = median_filter(f, [radius, radius])
        res[k] = cp.asnumpy(cp.where(cp.abs(f - med) > med * threshold, med, f))
    return res

def flat_field_divide(frames, flat_mean):
    fmean = float(np.asarray(flat_mean).mean()) + 1e-12
    return np.nan_to_num(np.asarray(frames, 'float32') / fmean, nan=1.0)

def bin2(a, nbin):
    for _ in range(nbin):
        a = 0.5 * (a[..., 0::2, :] + a[..., 1::2, :])
        a = 0.5 * (a[..., :, 0::2] + a[..., :, 1::2])
    return a

## Geometry and angle selection (same cell as step 1)

In [ ]:
with h5py.File(GRID_H5, 'r') as f:
    NANG, NY, NX = f[ORCA].shape
cy, cx = NY // 2, NX // 2
SL = (slice(cy - CROP[0] // 2, cy + CROP[0] // 2),
      slice(cx - CROP[1] // 2, cx + CROP[1] // 2))

with h5py.File(META_H5, 'r') as f:
    theta_all = np.asarray(f[THETA_DS], 'float64')
assert len(theta_all) == NANG, f'{len(theta_all)} angles vs {NANG} frames'

# ALL frames, always -- no angle subsampling at preprocessing time. Quick-look
# runs subsample later, at step 6, via config_step6.conf's own
# ntheta/start_theta/stride (CodedReader indexes directly into this fully
# stored array). Same convention AtomiumS2_coded uses: write everything once,
# select a subset only when solving.
NTHETA = NANG
sel = np.arange(NANG)
theta_deg = theta_all[sel]

mag       = Z_DET / Z1
voxelsize = DET_PIX / mag   # NATIVE voxel size -- no bin factor, steps 1-5 never bin

print(f'orca {NY} x {NX}, {NANG} frames; crop rows {SL[0].start}:{SL[0].stop} '
      f'cols {SL[1].start}:{SL[1].stop}')
print(f'theta: ALL {NTHETA} angles converted, [{theta_deg.min():.3f}, {theta_deg.max():.3f}] deg, '
      f'step {np.diff(theta_deg).mean():.4f} deg')
print(f'voxel {voxelsize*1e9:.2f} nm   magnification {mag:.3f}')
print('quick-look subsampling, if wanted, happens at step 6 via '
      "config_step6.conf's own ntheta/start_theta/stride")

plt.figure(figsize=(5, 3))
plt.plot(theta_all, '.', ms=2)
plt.xlabel('frame'); plt.ylabel('tom_ry [deg]'); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Stream the projections

In [ ]:
# Written by step1 into the same HDF5.
with h5py.File(OUT_H5, 'r') as f:
    dark      = np.asarray(f['/exchange/dark'], 'float32')
    flat_norm = np.asarray(f['/exchange/flat_norm'], 'float32')
    dead_mask = np.asarray(f['/exchange/dead_mask']).astype(bool)
    flat_mean = float(f['/exchange/flat_mean'][0])
print(f'dark/flat/dead_mask from {OUT_H5}   flat_mean {flat_mean:.1f} ADU')

# pdata goes in UNBINNED, at the crop size, for ALL NTHETA frames -- no angle
# subsampling here (see GEOM_THETA above). CodedReader.read_data bins it by
# 2**bin on the way out, so pre-binning here would bin it TWICE.
#
# That is also what makes the bin ladder AND the step-6 angle ladder work:
# any config_step6.conf ntheta/start_theta/stride, and any bin, read the same
# file. The cost is size -- NTHETA x 2048 x 2048 float32, ~63 GB at the full
# 4001 angles.
CY, CX = CROP
CHUNK = 16
with h5py.File(OUT_H5, 'a') as fo:
    if '/exchange/pdata' in fo:
        del fo['/exchange/pdata']
    ds = fo.create_dataset('/exchange/pdata', shape=(NTHETA, CY, CX),
                           dtype='float32', chunks=(1, CY, CX))
    with h5py.File(GRID_H5, 'r') as fi:
        for cs in range(0, NTHETA, CHUNK):
            ce = min(cs + CHUNK, NTHETA)
            raw = np.asarray(fi[ORCA][sel[cs:ce], SL[0], SL[1]], 'float32')
            raw = dark_subtract_clip(raw, dark)
            raw = fill_dead(raw, dead_mask)
            if DEZINGER:
                raw = remove_outliers(raw, *DEZINGER)
            ds[cs:ce] = flat_field_divide(raw, flat_mean)
            if cs % (CHUNK * 10) == 0:
                print(f'  {ce}/{NTHETA}', flush=True)
    # pref is NOT read by CodedReader; step3 uses it for registration only,
    # so it is stored unbinned too for consistency.
    if '/exchange/pref' in fo:
        del fo['/exchange/pref']
    fo.create_dataset('/exchange/pref', data=flat_norm)
gb = NTHETA * CY * CX * 4 / 1024 ** 3
print(f'wrote /exchange/pdata {(NTHETA, CY, CX)} UNBINNED  ({gb:.1f} GB)')
print(f'   step6.py/CodedReader will bin it by 2**{CFG.bin} -> {CFG.nz} x {CFG.n}')

## Diagnostic: do the projections look like a rotating sample?

First, middle and last angle, plus their difference. The CA2 speckle should be
visible on every frame and the sample should visibly rotate between them. If
first and last look identical, `tom_ry` and the frame order disagree.

In [ ]:
# Binned here ONLY for a smaller display thumbnail -- on disk, and for every
# registration computation in step 3, pdata stays native/unbinned.
with h5py.File(OUT_H5, 'r') as f:
    p0 = bin2(np.asarray(f['/exchange/pdata'][0])[None], PREVIEW_BIN)[0]
    pm = bin2(np.asarray(f['/exchange/pdata'][NTHETA // 2])[None], PREVIEW_BIN)[0]
    pl = bin2(np.asarray(f['/exchange/pdata'][NTHETA - 1])[None], PREVIEW_BIN)[0]
stack = np.stack([p0, pm, pl])
print(f'pdata range [{stack.min():.4f}, {stack.max():.4f}]  mean {stack.mean():.4f}')

fig, ax = plt.subplots(1, 4, figsize=(20, 3.8))
cbshow(ax[0], p0,  f'theta {theta_deg[0]:.0f} deg')
cbshow(ax[1], pm,  f'theta {theta_deg[NTHETA//2]:.0f} deg')
cbshow(ax[2], pl,  f'theta {theta_deg[-1]:.0f} deg')
cbshow(ax[3], p0 - pl, 'first - last')
plt.tight_layout(); plt.show()